In [13]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")
print(df.shape)
print(df.columns.tolist())
df.head()

(891, 15)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [14]:
raw = df.copy()

def summary(d):
    return {
        "rows": len(d),
        "columns": d.shape[1],
        "total null values": int(d.isnull().sum().sum()),
        "duplicate rows": int(d.duplicated().sum()),
    }

report = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulls": df.isnull().sum(),
    "null %": (df.isnull().mean() * 100).round(1),
})
print("Duplicate rows:", df.duplicated().sum())
report

Duplicate rows: 107


,dtype,nulls,null %
survived,int64,0,0.0
pclass,int64,0,0.0
sex,object,0,0.0
age,float64,177,19.9
sibsp,int64,0,0.0
parch,int64,0,0.0
fare,float64,0,0.0
embarked,object,2,0.2
class,category,0,0.0
who,object,0,0.0


In [15]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [16]:
df = df.drop(columns="deck")
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
print("Nulls left:", df.isnull().sum().sum())

Nulls left: 0


In [17]:
dupes = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", dupes)
print("Rows now:", len(df))

Duplicate rows removed: 116
Rows now: 775


In [18]:
for col in ["sex", "who", "alive", "embark_town"]:
    df[col] = df[col].astype(str).str.strip().str.title()
df["embarked"] = df["embarked"].astype(str).str.strip().str.upper()
df["sex"].value_counts()

,count
sex,
Male,483
Female,292


In [19]:
outlier_info = {}
for col in ["age", "fare"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_info[col] = {"lower": round(lower, 2), "upper": round(upper, 2), "outliers": int(count)}
print(pd.DataFrame(outlier_info))

fare_upper = outlier_info["fare"]["upper"]
df["fare"] = df["fare"].clip(upper=fare_upper)

           age    fare
lower     -1.5  -31.17
upper     58.5   73.42
outliers  27.0  102.00


In [20]:
df["pclass"] = df["pclass"].astype("category")
df["survived"] = df["survived"].astype(bool)
for col in ["sex", "embarked", "who", "embark_town", "alive"]:
    df[col] = df[col].astype("category")
df["class"] = df["class"].astype(str)
df.dtypes

,0
survived,bool
pclass,category
sex,category
age,float64
sibsp,int64
parch,int64
fare,float64
embarked,category
class,object
who,category


In [21]:
comparison = pd.DataFrame({"Before": summary(raw), "After": summary(df)})
comparison

,Before,After
rows,891,775
columns,15,14
total null values,869,0
duplicate rows,107,6


In [22]:
from google.colab import files
df.to_csv("titanic_cleaned.csv", index=False)
files.download("titanic_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, json, shutil
os.chdir("/content")
from getpass import getpass
from google.colab import _message

token = getpass("Paste GitHub token: ").strip()
if not (token.startswith("ghp_") or token.startswith("github_pat_")) or len(token) > 120:
    raise SystemExit("This is not a GitHub token. Copy the token (starts with ghp_) and run the cell again.")

nb = _message.blocking_request('get_ipynb', timeout_sec=120)['ipynb']

!rm -rf /content/OIBSIP
!git clone https://{token}@github.com/Ali-Bin-Talib/OIBSIP.git /content/OIBSIP

folder = "/content/OIBSIP/DataAnalytics-L1-CleaningData"
os.makedirs(folder, exist_ok=True)

with open(folder + "/Cleaning_Data.ipynb", "w") as f:
    json.dump(nb, f)

shutil.copy("/content/titanic_cleaned.csv", folder + "/titanic_cleaned.csv")

readme = """# Cleaning Data (Titanic)

A data cleaning project that turns the raw Titanic dataset into an analysis-ready dataset, with every decision documented.

## What this project does
- Data quality report: nulls per column, duplicate rows, data types, value ranges
- Missing data handling with a justified strategy for each column
- Duplicate detection and removal
- Standardisation of text values (casing, whitespace)
- Outlier detection with the IQR method and a decision for each column
- Data type correction
- Before vs after summary table
- Cleaned dataset saved as titanic_cleaned.csv

## Tech Stack
Python, pandas, numpy, seaborn, Jupyter Notebook (Google Colab)

## How to Run
Open Cleaning_Data.ipynb in Google Colab and run all cells. The Titanic data is loaded with seaborn, so no download is needed.
"""
with open(folder + "/README.md", "w") as f:
    f.write(readme)

os.chdir("/content/OIBSIP")
!git config user.email "alibintalib853@gmail.com"
!git config user.name "Ali Bin Talib"
!git add .
!git commit -m "Data Analytics L1: Cleaning Data"
!git push
os.chdir("/content")

Paste GitHub token: ··········
